In [1]:
# Install FAISS
!pip install -q faiss-cpu

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Creating knowledge base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating FAISS index...")

model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = model.encode(
    kb,
    show_progress_bar=False,
    convert_to_numpy=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.7 MB/s eta 0:00:00
Creating knowledge base...
Loading embedding model and creating FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created!


In [2]:
# Zero-shot classifier setup for Q1, Q2 and Q6

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
#Question-1 :
"""Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the
row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted 
probability score assigned to the ground-truth correct option 
(option in the answer column)? (Round to 3 decimal points)"""

result = zs(
    prompt_150,
    candidate_labels=labels_150
)

correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print(round(correct_score, 3))

0.384


In [4]:
#Question-2:
"""Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your 
FAISS index to retrieve the top k=10 most similar documents. At what exact rank 
(1 through 10) did FAISS place the true correct document (which is the document 
originally located at index 150 in the KB)?"""

query_embedding = model.encode(
    [prompt_150],
    convert_to_numpy=True
)

distances, indices = index.search(query_embedding, k=10)

retrieved_indices = indices[0]

rank = list(retrieved_indices).index(150) + 1

print("Retrieved KB indices:", retrieved_indices)
print("Rank of true document:", rank)

Retrieved KB indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of true document: 10


In [5]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [6]:
#Question-3:
"""Q3. Take the top 10 documents retrieved by FAISS in the previous question. 
Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 
documents and sort them by the cross-encoder's score. At what exact rank 
(1 through 10) does the Cross-Encoder place the true correct document?"""

# Sort the retrieved documents by Cross-Encoder score (highest first)
sorted_indices = np.argsort(ce_scores)[::-1]

# Reorder the original FAISS indices according to the new ranking
reranked_indices = retrieved_indices[sorted_indices]

# Find the rank of the true document (KB index 150)
rank = list(reranked_indices).index(150) + 1

print("Reranked KB indices:", reranked_indices)
print("Cross-Encoder rank of true document:", rank)

Reranked KB indices: [ 150 1906  847 1693 1269 1532  168  576 1701  663]
Cross-Encoder rank of true document: 1


In [7]:
#Question-4:
"""Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate
them with a single space between each. Create a string: "Context: 
[concatenated_docs] Question: [prompt]". Tokenize this string using the 
bert-base-uncased tokenizer (without truncation). Exactly how many total tokens 
does this generate?"""

# Get row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode(
    [prompt_42],
    convert_to_numpy=True
)

# Retrieve top 5 documents
_, indices = index.search(query_embedding, k=5)

retrieved_docs = [kb[i] for i in indices[0]]

# Concatenate documents with a single space
context = " ".join(retrieved_docs)

# Create the final input string
input_text = f"Context: {context} Question: {prompt_42}"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize without truncation
tokens = tokenizer(input_text, truncation=False)

# Count total tokens
print("Total tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


In [8]:
#Question-5:
"""Q5. Retrieve the exact true document for row index 150 from your KB. Create a 
RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot
classification from Question 1 on this augmented string. What is the new predicted
probability score of the ground-truth correct option? (Round to 3 decimal places)."""

# Retrieve the true document
true_document = kb[150]

# Create the RAG input
rag_input = f"Context: {true_document} Question: {prompt_150}"

# Run zero-shot classification
result = zs(
    rag_input,
    candidate_labels=labels_150
)

# Get the score of the correct answer
correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Ground-truth probability:", round(correct_score, 3))

Ground-truth probability: 0.989


In [9]:
#Question-6:
"""Q6. What happens if your vector database retrieves the wrong information? 
Take the prompt for row index 150. Manually force the context to be the document 
located at KB index 999 (a completely unrelated fact). Run the zero-shot 
classifier on this "Adversarial RAG" string. What is the probability of the 
correct option now? (Round to 3 decimal places).
"""

# Use an unrelated document from the knowledge base
wrong_document = kb[999]

# Create the Adversarial RAG input
adversarial_input = f"Context: {wrong_document} Question: {prompt_150}"

# Run the zero-shot classifier
result = zs(
    adversarial_input,
    candidate_labels=labels_150
)

# Get the probability of the correct answer
correct_score = result["scores"][
    result["labels"].index(ans_150)
]

print("Ground-truth probability:", round(correct_score, 3))

Ground-truth probability: 0.529


In [10]:
#Question-7:
"""In RAG, a "Hit" occurs if the retrieved context contains the facts needed to 
answer the question.
Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 
documents for each prompt. If the exact string of the row's correct option is 
found inside any of those 5 retrieved documents, it counts as a hit. What is the 
exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal 
place)."""

hits = 0

for i in range(100):
    row = train.iloc[i]

    prompt = str(row["prompt"])

    correct_text = str(row[row["answer"]])

    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    )

    _, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1))

Hits: 73
Hit Rate: 73.0


In [11]:
#Question-8:
"""Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of 
train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS 
Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 
Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] 
Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot 
classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from 
highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and 
calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline 
across these 20 rows? (Round to 3 decimal places)."""

scores = []

for i in range(20):
    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct_letter = row["answer"]

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    # -------------------------------
    # Step 1: Retrieve (Top-5)
    # -------------------------------
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    )

    _, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # -------------------------------
    # Step 2: Rerank
    # -------------------------------
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # -------------------------------
    # Step 3: Augment
    # -------------------------------
    rag_input = f"Context: {best_doc} Question: {prompt}"

    # -------------------------------
    # Step 4: Predict
    # -------------------------------
    result = zs(
        rag_input,
        candidate_labels=labels
    )

    # -------------------------------
    # Step 5: Convert labels to letters
    # -------------------------------
    predicted_letters = []

    for predicted_text in result["labels"][:3]:
        for letter in ["A", "B", "C", "D", "E"]:
            if predicted_text == str(row[letter]):
                predicted_letters.append(letter)
                break

    # MAP@3
    if correct_letter in predicted_letters:
        score = 1 / (predicted_letters.index(correct_letter) + 1)
    else:
        score = 0

    scores.append(score)

map3 = np.mean(scores)

print("Final MAP@3:", round(map3, 3))

Final MAP@3: 0.975
